# Chapter 17: Hierarchical Indexing

In [1]:
import numpy as np
import pandas as pd

## 17.1 A Multiply Indexed Series

Let’s start by considering how we might represent two-dimensional data within a
one-dimensional Series. For concreteness, we will consider a series of data where
each point has a character and numerical key.

### The Bad Way

In [2]:
index = [('California', 2010), ('California', 2020), 
         ('New York', 2010), ('New York', 2020), 
         ('Texas', 2010), ('Texas', 2020)]
population = [37253956, 39538223,
               19378102, 20201249,
               25145561, 29145505]
pop = pd.Series(population, index=index)
pop

(California, 2010)    37253956
(California, 2020)    39538223
(New York, 2010)      19378102
(New York, 2020)      20201249
(Texas, 2010)         25145561
(Texas, 2020)         29145505
dtype: int64

**이 방식의 문제점**: Tuple index로는 '2010년 자료만 선택' 같은 작업이 매우 불편함.

In [3]:
pop[('California', 2020):('Texas', 2010)]

(California, 2020)    39538223
(New York, 2010)      19378102
(New York, 2020)      20201249
(Texas, 2010)         25145561
dtype: int64

In [4]:
pop[[i for i in pop.index if i[1] == 2010]]

(California, 2010)    37253956
(New York, 2010)      19378102
(Texas, 2010)         25145561
dtype: int64

### **The Better Way**: The Pandas MultiIndex

In [5]:
# MultiIndex로 변환
index = pd.MultiIndex.from_tuples(index)
pop = pop.reindex(index)
print(pop)

California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64


Now to access all data for which the second index is 2020, we can use the Pandas slic‐
ing notation:

In [6]:
pop[:, 2020]

California    39538223
New York      20201249
Texas         29145505
dtype: int64

## 17.2 MultiIndex as Extra Dimension

unstack() 메서드를 사용하면 MultiIndex Series를 일반 DataFrame으로 변환할 수 있다.

In [7]:
# unstack(): index를 column로 변환
pop_df = pop.unstack()
print(pop_df)

                2010      2020
California  37253956  39538223
New York    19378102  20201249
Texas       25145561  29145505


In [8]:
# stack(): column을 index로 변환 (unstack의 반대)
print(pop_df.stack())

California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64


In [9]:
pop_df = pd.DataFrame({'total': pop,
                       'under18': [9284094, 8898092,
                                   4318033, 4181528,
                                   6879014, 7432474]})
pop_df

total  under18
California 2010  37253956  9284094
           2020  39538223  8898092
New York   2010  19378102  4318033
           2020  20201249  4181528
Texas      2010  25145561  6879014
           2020  29145505  7432474

In [10]:
f_u18 = pop_df['under18'] / pop_df['total']
f_u18.unstack()

,2010,2020
California,0.249211,0.225050
New York,0.222831,0.206994
Texas,0.273568,0.255013


## 17.3 Methods of MultiIndex Creation 

### Method 1: Creation from list

In [11]:
df = pd.DataFrame(np.random.rand(4, 2), 
                  index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                  columns=['data1', 'data2'])
df

data1     data2
a 1  0.143125  0.101172
  2  0.083621  0.561722
b 1  0.096747  0.932666
  2  0.056171  0.061083

### Method 2: from Tuple dictionary

In [12]:
data = {('California', 2010): 37253956,
        ('California', 2020): 39538223,
        ('New York', 2010): 19378102,
        ('New York', 2020): 20201249,
        ('Texas', 2010): 25145561,
        ('Texas', 2020): 29145505}
print(pd.Series(data))

California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64


### Method 3: 명시적 MultiIndex 생성자

In [13]:
# from_arrays
pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]])

# from_tuples
pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)])

# from_product (카테시안 곱)
pd.MultiIndex.from_product([['a', 'b'], [1, 2]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

### MultiIndex Level Names

In [14]:
pop.index.names = ['state', 'year']
pop

state       year
California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

## 17.4 MultiIndex for Columns

**Column** of DataFrame can have MultiIndex

In [15]:
# row and column all MultiIndex Creation
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'],
                                      ['HR', 'Temp']],
                                     names=['subject', 'type'])

# Creation data
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] = 10 * data[:, ::2]
data += 37

health_data = pd.DataFrame(data, index, columns=columns)
print(health_data)

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      18.0  37.9  45.0  35.5  49.0  35.5
     2      50.0  36.4  46.0  37.1  35.0  34.6
2014 1      46.0  37.1  29.0  36.9  45.0  36.4
     2      50.0  35.2  47.0  37.4  47.0  37.9


In [16]:
# Guido의 모든 data 선택
print(health_data['Guido'])

type          HR  Temp
year visit            
2013 1      45.0  35.5
     2      46.0  37.1
2014 1      29.0  36.9
     2      47.0  37.4


## 17.5 Indexing and Slicing a MultiIndex

### Multiply Indexed Series

In [17]:
# access single element
print(pop['California', 2010])

# sub indexing (only first level)
print(pop['California'])

# slicing
print(pop.loc['California':'New York'])

# sub indexing (second level)
print(pop[:, 2010])

37253956
year
2010    37253956
2020    39538223
dtype: int64
state       year
California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
dtype: int64
state
California    37253956
New York      19378102
Texas         25145561
dtype: int64


### Multiply Indexed DataFrames

In [18]:
health_data['Guido', 'HR']

year  visit
2013  1        45.0
      2        46.0
2014  1        29.0
      2        47.0
Name: (Guido, HR), dtype: float64

In [19]:
health_data.iloc[:2, :2]

subject      Bob      
type          HR  Temp
year visit            
2013 1      18.0  37.9
     2      50.0  36.4

In [20]:
health_data.loc[:, ('Bob', 'HR')]

year  visit
2013  1        18.0
      2        50.0
2014  1        46.0
      2        50.0
Name: (Bob, HR), dtype: float64

In [21]:
idx = pd.IndexSlice
health_data.loc[idx[:, 1], idx[:, 'HR']]

,subject,Bob,Guido,Sue
,type,HR,HR,HR
year,visit,,,
2013,1,18.0,45.0,49.0
2014,1,46.0,29.0,45.0


## 17.6 Rearranging Multi-Indexes

### Sorted and Unsorted Indices

In [22]:
# Creating unsorted MultiIndex
index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']

# Unsorted array - slicing error
# data['a':'b'] # KeyError!

# by sort_index() sorting
data = data.sort_index()
print(data['a':'b'])

char  int
a     1      0.100895
      2      0.240912
b     1      0.984752
      2      0.814215
dtype: float64


### Stacking and Unstacking Indices

In [23]:
# unstack: index to column
print(pop.unstack(level=0))

state  California  New York     Texas
year                                 
2010     37253956  19378102  25145561
2020     39538223  20201249  29145505


In [24]:
# unstack: another level
print(pop.unstack(level=1))

year            2010      2020
state                         
California  37253956  39538223
New York    19378102  20201249
Texas       25145561  29145505


### Index Setting and Resetting

In [25]:
# reset_index: change index to column
pop_flat = pop.reset_index(name='population')
pop_flat

,state,year,population
0,California,2010,37253956
1,California,2020,39538223
2,New York,2010,19378102
3,New York,2020,20201249
4,Texas,2010,25145561
5,Texas,2020,29145505


In [26]:
# set_index: change column to index
print(pop_flat.set_index(['state', 'year']))

                 population
state      year            
California 2010    37253956
           2020    39538223
New York   2010    19378102
           2020    20201249
Texas      2010    25145561
           2020    29145505
